In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline

In [2]:
# -----------------------------------
# 1. 데이터 불러오기
# -----------------------------------
df = pd.read_csv("../datasets/ecommerce_customer_data.csv")

# 데이터 확인
print("데이터 크기:", df.shape)
print(df.head())

데이터 크기: (2000, 9)
   Customer_ID  Age  Gender  Annual_Income  Spending_Score  Membership_Years  \
0            1   56    Male          69812              88               3.2   
1            2   69  Female          70500              26               4.3   
2            3   46  Female          99151              17               8.2   
3            4   32    Male          78643              71               0.6   
4            5   60  Female          64900              13               6.7   

   Online_Purchases  Discount_Usage  Churn  
0                92            0.43      1  
1                30            0.23      1  
2               199            0.52      0  
3               153            0.25      0  
4               127            0.94      0  


Age: 나이
Gender: 성별
Annual_Income: 연소득
Spending_Score: 소비 성향 점수
Membership_Years: 멤버십 유지 기간
Online_Purchases: 온라인 구매 횟수
Discount_Usage: 할인 사용 정도
Churn: 이탈 여부

Customer_ID는 식별자라 제거해야 함
클래스가 약간 불균형
Churn=0: 1379
Churn=1: 621
수치형 변수들 간 단순 선형 상관이 강하지 않음
즉 로지스틱 회귀 하나로는 분리가 잘 안 될 수도 있고, 트리 계열이 더 나을 가능성이 있습니다.

In [3]:
# -----------------------------------
# 2. 입력(X), 타깃(y) 분리
# -----------------------------------
X = df.drop(columns=["Customer_ID", "Churn"])
y = df["Churn"]

# -----------------------------------
# 3. 범주형 변수 인코딩
# Gender를 숫자로 변환
# -----------------------------------
X = pd.get_dummies(X, columns=["Gender"], drop_first=True)

print("\n전처리 후 입력 데이터 컬럼:")
print(X.columns)

# -----------------------------------
# 4. train / val / test 분리
# stratify=y 로 클래스 비율 유지
# -----------------------------------

# 1차 분할: train 70%, temp 30%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# 2차 분할: temp 30%를 val 15%, test 15%로 반반 분할
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

total = len(X)

print("\n" + "="*50)
print("데이터 분할 결과")
print("="*50)
print(f"전체 데이터 수      : {total}")
print(f"훈련 데이터 (train): X={X_train.shape}, y={y_train.shape}, 비율={len(X_train)/total:.1%}")
print(f"검증 데이터 (val)  : X={X_val.shape}, y={y_val.shape}, 비율={len(X_val)/total:.1%}")
print(f"테스트 데이터 (test): X={X_test.shape}, y={y_test.shape}, 비율={len(X_test)/total:.1%}")

# -----------------------------------
# 5. Logistic Regression용 스케일링
# 트리 계열은 스케일링 없이도 됨
# -----------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


전처리 후 입력 데이터 컬럼:
Index(['Age', 'Annual_Income', 'Spending_Score', 'Membership_Years',
       'Online_Purchases', 'Discount_Usage', 'Gender_Male'],
      dtype='str')

데이터 분할 결과
전체 데이터 수      : 2000
훈련 데이터 (train): X=(1400, 7), y=(1400,), 비율=70.0%
검증 데이터 (val)  : X=(300, 7), y=(300,), 비율=15.0%
테스트 데이터 (test): X=(300, 7), y=(300,), 비율=15.0%


In [4]:
# -----------------------------------
# 6. 모델 생성
# -----------------------------------
log_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42,
)

tree_model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    criterion="gini",
    random_state=42,
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=7,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
)

In [8]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate_model(model_name, y_true, y_pred, split_name):
    print("\n" + "="*60)
    print(f"{model_name} | {split_name} 결과")
    print("="*60)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("Classification Report:\n", classification_report(y_true, y_pred))

In [9]:
# -----------------------------------
# 7. Logistic Regression 학습 및 평가
# -----------------------------------
log_model.fit(X_train_scaled, y_train)

# 예측
y_pred_log_train = log_model.predict(X_train_scaled)
y_pred_log_val = log_model.predict(X_val_scaled)
y_pred_log_test = log_model.predict(X_test_scaled)

# 평가
evaluate_model("Logistic Regression", y_train, y_pred_log_train, "Train")
evaluate_model("Logistic Regression", y_val, y_pred_log_val, "Validation")
evaluate_model("Logistic Regression", y_test, y_pred_log_test, "Test")


Logistic Regression | Train 결과
Accuracy: 0.6892857142857143
Confusion Matrix:
 [[965   0]
 [435   0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.69      1.00      0.82       965
           1       0.00      0.00      0.00       435

    accuracy                           0.69      1400
   macro avg       0.34      0.50      0.41      1400
weighted avg       0.48      0.69      0.56      1400


Logistic Regression | Validation 결과
Accuracy: 0.69
Confusion Matrix:
 [[207   0]
 [ 93   0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.69      1.00      0.82       207
           1       0.00      0.00      0.00        93

    accuracy                           0.69       300
   macro avg       0.34      0.50      0.41       300
weighted avg       0.48      0.69      0.56       300


Logistic Regression | Test 결과
Accuracy: 0.69
Confusion Matrix:
 [[207   0]
 [ 93   0]]
Classificatio

c:\Users\Playdata\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Playdata\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Playdata\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [10]:
# -----------------------------------
# 8. Decision Tree 학습 및 평가
# -----------------------------------
tree_model.fit(X_train, y_train)

# 예측
y_pred_tree_train = tree_model.predict(X_train)
y_pred_tree_val = tree_model.predict(X_val)
y_pred_tree_test = tree_model.predict(X_test)

# 평가
evaluate_model("Decision Tree", y_train, y_pred_tree_train, "Train")
evaluate_model("Decision Tree", y_val, y_pred_tree_val, "Validation")
evaluate_model("Decision Tree", y_test, y_pred_tree_test, "Test")


Decision Tree | Train 결과
Accuracy: 0.6964285714285714
Confusion Matrix:
 [[963   2]
 [423  12]]
Classification Report:
               precision    recall  f1-score   support

           0       0.69      1.00      0.82       965
           1       0.86      0.03      0.05       435

    accuracy                           0.70      1400
   macro avg       0.78      0.51      0.44      1400
weighted avg       0.75      0.70      0.58      1400


Decision Tree | Validation 결과
Accuracy: 0.6833333333333333
Confusion Matrix:
 [[204   3]
 [ 92   1]]
Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.99      0.81       207
           1       0.25      0.01      0.02        93

    accuracy                           0.68       300
   macro avg       0.47      0.50      0.42       300
weighted avg       0.55      0.68      0.57       300


Decision Tree | Test 결과
Accuracy: 0.6833333333333333
Confusion Matrix:
 [[204   3]
 [ 92   1]]
Cla

In [12]:
# -----------------------------------
# 9. Random Forest 학습 및 평가
# -----------------------------------
rf_model.fit(X_train, y_train)

# 예측
y_pred_rf_train = rf_model.predict(X_train)
y_pred_rf_val = rf_model.predict(X_val)
y_pred_rf_test = rf_model.predict(X_test)

# 평가
evaluate_model("Random Forest", y_train, y_pred_rf_train, "Train")
evaluate_model("Random Forest", y_val, y_pred_rf_val, "Validation")
evaluate_model("Random Forest", y_test, y_pred_rf_test, "Test")


Random Forest | Train 결과
Accuracy: 0.7057142857142857
Confusion Matrix:
 [[965   0]
 [412  23]]
Classification Report:
               precision    recall  f1-score   support

           0       0.70      1.00      0.82       965
           1       1.00      0.05      0.10       435

    accuracy                           0.71      1400
   macro avg       0.85      0.53      0.46      1400
weighted avg       0.79      0.71      0.60      1400


Random Forest | Validation 결과
Accuracy: 0.6833333333333333
Confusion Matrix:
 [[205   2]
 [ 93   0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.69      0.99      0.81       207
           1       0.00      0.00      0.00        93

    accuracy                           0.68       300
   macro avg       0.34      0.50      0.41       300
weighted avg       0.47      0.68      0.56       300


Random Forest | Test 결과
Accuracy: 0.6866666666666666
Confusion Matrix:
 [[206   1]
 [ 93   0]]
Cla

In [13]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Logistic Regression", "Logistic Regression",
              "Decision Tree", "Decision Tree", "Decision Tree",
              "Random Forest", "Random Forest", "Random Forest"],
    "Split": ["Train", "Validation", "Test",
              "Train", "Validation", "Test",
              "Train", "Validation", "Test"],
    "Accuracy": [
        accuracy_score(y_train, y_pred_log_train),
        accuracy_score(y_val, y_pred_log_val),
        accuracy_score(y_test, y_pred_log_test),

        accuracy_score(y_train, y_pred_tree_train),
        accuracy_score(y_val, y_pred_tree_val),
        accuracy_score(y_test, y_pred_tree_test),

        accuracy_score(y_train, y_pred_rf_train),
        accuracy_score(y_val, y_pred_rf_val),
        accuracy_score(y_test, y_pred_rf_test),
    ]
})

print(results)

                 Model       Split  Accuracy
0  Logistic Regression       Train  0.689286
1  Logistic Regression  Validation  0.690000
2  Logistic Regression        Test  0.690000
3        Decision Tree       Train  0.696429
4        Decision Tree  Validation  0.683333
5        Decision Tree        Test  0.683333
6        Random Forest       Train  0.705714
7        Random Forest  Validation  0.683333
8        Random Forest        Test  0.686667


In [65]:
voting_model = VotingClassifier(
    estimators=[
        ("lr", log_model),
        ("dt", tree_model),
        ("rf", rf_model)
    ],
    voting="soft"
)

voting_model.fit(X_train, y_train)
y_prob_vote = voting_model.predict_proba(X_test)[:, 1]
y_pred_vote = (y_prob_vote >= 0.7).astype(int)

In [66]:
print("\n" + "="*50)
print("Voting model 결과")
print("="*50)
print("Accuracy:", accuracy_score(y_test, y_pred_vote))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_vote))
print("Classification Report:\n", classification_report(y_test, y_pred_vote))


Voting model 결과
Accuracy: 0.69
Confusion Matrix:
 [[276   0]
 [124   0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.69      1.00      0.82       276
           1       0.00      0.00      0.00       124

    accuracy                           0.69       400
   macro avg       0.34      0.50      0.41       400
weighted avg       0.48      0.69      0.56       400



c:\Users\Playdata\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Playdata\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Playdata\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [68]:
# 저장
with open("../artifacts/voting_model.pkl", "wb") as f:
    pickle.dump(voting_model, f)

In [69]:
# 불러오기
with open("../artifacts/voting_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)